# 01 · Preparación e integración de los datos

**Proyecto 2 · El costo invisible del progreso · DATA-1002**

Este notebook toma los cuatro archivos originales de `data/raw/`, los revisa, los limpia y los integra en una sola tabla país-año: `data/processed/datos_integrados.csv`. Esa tabla alimenta el notebook de análisis (`02_analisis.ipynb`) y el dashboard (`src/dashboard_desacoplamiento.py`).

**Pregunta del proyecto:** ¿En qué medida cambia la identificación de países que han logrado desacoplar su crecimiento económico de las emisiones de CO₂ al comparar las emisiones territoriales con las asociadas al consumo?

Las librerías se instalan con `pip install -r requirements.txt` desde la raíz del proyecto.

In [ ]:
import pandas as pd
import numpy as np

## 1. Fuentes

| Archivo | Contenido | Productor original | Unidad | Cobertura |
|---|---|---|---|---|
| `gdp-worldbank.csv` | PIB real, PPA | Banco Mundial, Eurostat, OCDE y FMI (2026), vía Our World in Data | dólares internacionales de 2021 | 1990–2025 |
| `territorial.csv` | Emisiones anuales de CO₂ dentro del país | Global Carbon Budget (2026), vía Our World in Data | toneladas de CO₂ | 1750–2025 |
| `consumo.csv` | Emisiones anuales de CO₂ asociadas al consumo (territorial − exportaciones + importaciones) | Global Carbon Budget (2026), vía Our World in Data | toneladas de CO₂ | 1990–2025 |
| `poblacion.csv` | Población | HYDE (2023), Gapminder (2022), UN WPP (2024), vía Our World in Data | personas | hasta 2023 |

Todas se descargaron como CSV desde [Our World in Data](https://ourworldindata.org/co2-emissions) (licencia CC BY 4.0). Según el profesor, tres o más archivos de una misma página son válidos como fuentes distintas.

**Granularidad:** en los cuatro archivos, una fila = una entidad (país o agregado) en un año. La llave de integración es compuesta: código ISO-3 + año.

## 2. Revisión inicial

In [ ]:
df_pib = pd.read_csv("../data/raw/gdp-worldbank.csv")
df_pib.head()

In [ ]:
df_pib.info()

In [ ]:
df_pib.isnull().sum()

In [ ]:
df_pib.shape

In [ ]:
print("Primer año:", df_pib.Year.min())
print("Último año:", df_pib.Year.max())

In [ ]:
df_pib.duplicated(subset=["Code", "Year"]).sum()

In [ ]:
df_territorial = pd.read_csv("../data/raw/territorial.csv")
df_consumo = pd.read_csv("../data/raw/consumo.csv")
df_poblacion = pd.read_csv("../data/raw/poblacion.csv")

In [ ]:
df_territorial.head()

In [ ]:
df_consumo.head()

In [ ]:
df_poblacion.head()

In [ ]:
# Diagnóstico mínimo de las otras tres fuentes (Sesión 4)
for nombre, df in [("Territorial", df_territorial), ("Consumo", df_consumo), ("Población", df_poblacion)]:
    valor = df.columns[-1]
    print(f"{nombre}: {df.shape} | faltantes en Code: {df['Code'].isna().sum()} | "
          f"faltantes en el valor: {df[valor].isna().sum()} | "
          f"duplicados Entity-Year: {df.duplicated(['Entity', 'Year']).sum()} | mínimo: {df[valor].min():,.0f}")

**Lectura:** no hay valores faltantes en las variables principales ni duplicados entidad-año. Algunos registros de emisiones valen 0 (entidades pequeñas o históricas); el análisis solo compara países con valores positivos en el período inicial. Los archivos sí mezclan países con **agregados** (continentes, grupos de ingreso, regiones del Banco Mundial) que tienen códigos `OWID_*`, `WB_*` o no tienen código: si se dejan, un país se contaría dos veces dentro de su región.

## 3. Limpieza

1. Renombrar las columnas a nombres cortos en español.
2. Conservar solo países: código ISO alfa-3 (tres letras mayúsculas).

In [ ]:
df_pib = df_pib.rename(columns={"Entity":"pais", "Code":"codigo", "Year":"anio", "GDP":"pib"})

df_territorial = df_territorial.rename(columns={"Entity":"pais", "Code":"codigo", "Year":"anio", "Annual CO₂ emissions":"co2_territorial"})

df_consumo = df_consumo.rename(columns={"Entity":"pais", "Code":"codigo", "Year":"anio", "Annual consumption-based CO₂ emissions":"co2_consumo"})

df_poblacion = df_poblacion.rename(columns={"Entity":"pais", "Code":"codigo", "Year":"anio", "Population":"poblacion"})

In [ ]:
print("PIB:", df_pib.shape)
print("Territorial:", df_territorial.shape)
print("Consumo:", df_consumo.shape)
print("Población:", df_poblacion.shape)

In [ ]:
print("PIB:", df_pib.anio.min(), "-", df_pib.anio.max())
print("Territorial:", df_territorial.anio.min(), "-", df_territorial.anio.max())
print("Consumo:", df_consumo.anio.min(), "-", df_consumo.anio.max())
print("Población:", df_poblacion.anio.min(), "-", df_poblacion.anio.max())

In [ ]:
filas_antes = {"PIB": len(df_pib), "Territorial": len(df_territorial), "Consumo": len(df_consumo), "Población": len(df_poblacion)}

In [ ]:
df_pib = df_pib[df_pib.codigo.str.fullmatch("[A-Z]{3}", na=False)].copy()

df_territorial = df_territorial[df_territorial.codigo.str.fullmatch("[A-Z]{3}", na=False)].copy()

df_consumo = df_consumo[df_consumo.codigo.str.fullmatch("[A-Z]{3}", na=False)].copy()

df_poblacion = df_poblacion[df_poblacion.codigo.str.fullmatch("[A-Z]{3}", na=False)].copy()

In [ ]:
filas_despues = {"PIB": len(df_pib), "Territorial": len(df_territorial), "Consumo": len(df_consumo), "Población": len(df_poblacion)}
pd.DataFrame({"filas originales": filas_antes, "filas de países": filas_despues}).assign(
    **{"filas excluidas (agregados)": lambda t: t["filas originales"] - t["filas de países"]})

In [ ]:
print("Duplicados PIB:", df_pib.duplicated(subset=["codigo", "anio"]).sum())

print("Duplicados territorial:", df_territorial.duplicated(subset=["codigo", "anio"]).sum())

print("Duplicados consumo:", df_consumo.duplicated(subset=["codigo", "anio"]).sum())

print("Duplicados población:", df_poblacion.duplicated(subset=["codigo", "anio"]).sum())

## 4. Integración (Sesión 10)

| Pregunta | Respuesta |
|---|---|
| ¿Qué representa cada fila? | Un país en un año, en las cuatro tablas. |
| ¿Cuál es la llave? | `codigo` + `anio`, con el mismo formato en todas. |
| ¿Hay duplicados en la llave? | No (celda anterior). |
| ¿Qué cardinalidad se espera? | 1 a 1; se valida con `validate="1:1"`. |
| ¿Qué tipo de unión? | `left` desde el PIB, para ver primero qué datos faltan; luego se eliminan los país-año sin las cuatro variables. |

In [ ]:
pib_info = df_pib[["pais", "codigo", "anio", "pib"]]

territorial_info = df_territorial[["codigo", "anio", "co2_territorial"]]

consumo_info = df_consumo[["codigo", "anio", "co2_consumo"]]

poblacion_info = df_poblacion[["codigo", "anio", "poblacion"]]

In [ ]:
df_union1 = pib_info.merge(territorial_info, on=["codigo", "anio"], how="left", validate="1:1")

df_union2 = df_union1.merge(consumo_info, on=["codigo", "anio"], how="left", validate="1:1")

df_final = df_union2.merge(poblacion_info, on=["codigo", "anio"], how="left", validate="1:1")

df_final.head()

In [ ]:
print("PIB original:", pib_info.shape)
print("Primera unión:", df_union1.shape)
print("Segunda unión:", df_union2.shape)
print("Unión final:", df_final.shape)

print("Duplicados:", df_final.duplicated(subset=["codigo", "anio"]).sum())

In [ ]:
df_final.isna().sum()

**Lectura:** las uniones no duplicaron filas (6.959 antes y después). Los faltantes muestran la cobertura de cada fuente: el CO₂ de consumo solo existe para unos 120 países, por eso es la variable con más vacíos. A continuación se compara cuántas filas deja cada tipo de unión entre PIB y CO₂ territorial.

In [ ]:
union_inner = pib_info.merge(territorial_info, on=["codigo", "anio"], how="inner", validate="1:1")

union_outer = pib_info.merge(territorial_info, on=["codigo", "anio"], how="outer", validate="1:1")

comparacion = pd.DataFrame({"tipo_union":["inner", "left", "outer"], "numero_filas":[union_inner.shape[0], df_union1.shape[0], union_outer.shape[0]]})

comparacion

## 5. Registros sin las cuatro variables

Se conservan solo los país-año con PIB, CO₂ territorial, CO₂ de consumo y población. Es una decisión necesaria para la pregunta: sin las cuatro variables no se puede comparar el crecimiento con las dos contabilidades. Tiene un costo que se reporta como limitación: quedan fuera los países sin datos de consumo, que son sobre todo países pequeños y de bajos ingresos.

In [ ]:
print("Registros antes de limpiar:", df_final.shape[0])

df_final = df_final.dropna(subset=["pib", "co2_territorial", "co2_consumo", "poblacion"]).copy()

df_final = df_final[(df_final.pib > 0) & (df_final.poblacion > 0)].copy()

print("Registros después de limpiar:", df_final.shape[0])

In [ ]:
excluidos = sorted(set(df_pib["codigo"]) - set(df_final["codigo"]))
print(f"Países con PIB que quedan fuera: {len(excluidos)}")
print("Con consumo pero sin PIB (no entran al panel):",
      sorted(set(df_consumo["codigo"]) - set(df_pib["codigo"])))

## 6. Variables derivadas

| Variable | Fórmula | Unidad |
|---|---|---|
| `territorial_per_capita`, `consumo_per_capita` | CO₂ / población | t CO₂ por persona |
| `intensidad_territorial`, `intensidad_consumo` | CO₂ / PIB × 1.000 | t CO₂ por cada 1.000 dólares (= kg por dólar) |
| `brecha_emisiones` | CO₂ consumo − CO₂ territorial | t CO₂ (positivo = importador neto de emisiones) |

In [ ]:
df_final["territorial_per_capita"] = df_final.co2_territorial / df_final.poblacion

df_final["consumo_per_capita"] = df_final.co2_consumo / df_final.poblacion

df_final["intensidad_territorial"] = df_final.co2_territorial / df_final.pib * 1000

df_final["intensidad_consumo"] = df_final.co2_consumo / df_final.pib * 1000

df_final["brecha_emisiones"] = df_final.co2_consumo - df_final.co2_territorial

In [ ]:
# Control de exactitud (Sesión 8): ¿hay país-año con el consumo idéntico al territorial?
consumo_copiado = np.isclose(df_final.co2_consumo, df_final.co2_territorial, rtol=1e-6, atol=0)

df_final[consumo_copiado].groupby(["codigo", "pais"])["anio"].agg(desde="min", hasta="max", años="count")

**Hallazgo de calidad:** en Bélgica, Luxemburgo, Sudáfrica, Namibia y Botsuana, el CO₂ de consumo de los años 90 es una **copia** del territorial. El Global Carbon Budget no tenía datos de comercio para esos años y repitió el valor. No se borran ni se imputan, para no cambiar la base que usa el dashboard; se documentan como limitación. Cuando el año inicial del período cae en los años 90, el cambio del CO₂ de consumo de esos países no es confiable (el notebook de análisis muestra cuánto cambia el resultado si se excluyen).

In [ ]:
df_final = df_final.sort_values(["pais", "anio"]).reset_index(drop=True)

df_final.head()

## 7. Resultado de la integración

In [ ]:
print("Cantidad de países:", df_final.codigo.nunique())
print("Cantidad de registros:", df_final.shape[0])
print("Primer año:", df_final.anio.min())
print("Último año:", df_final.anio.max())

df_final.groupby("anio")["codigo"].nunique()

El panel final tiene **3.995 filas (país-año), 118 países, 1990–2023**. El año final es 2023 porque la población llega hasta ese año.

In [ ]:
df_final.to_csv("../data/processed/datos_integrados.csv", index=False)
print("Guardado: data/processed/datos_integrados.csv", df_final.shape)